# 1. The datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/01_data.ipynb)

Loads every dataset in `DATA/` and checks that each file is identical to the file the analyses read.

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

In [3]:
run("python DATA/build_data.py --check")

DATA/ check: every file matches its source
[exit 0, 0 s]


'DATA/ check: every file matches its source\n'

### Octane isomers (Table 4, Supplementary Table S5)

In [4]:
oct = pd.read_csv("DATA/octane_properties.csv")
display(oct) if "display" in dir() else print(oct)
check("number of octane isomers", 18, len(oct))
check("dHvap of 2,2,3,3-tetramethylbutane used (kcal/mol)", 8.41, float(oct.set_index("name").loc["2,2,3,3-tetramethylbutane", "dHvap_kcal_per_mol"]))

                         name            smiles  T_B_C  dHf_kcal_per_mol  \
0                      octane          CCCCCCCC  125.6            -49.82   
1             2-methylheptane        CC(C)CCCCC  117.6            -51.50   
2             3-methylheptane        CCC(C)CCCC  118.9            -50.82   
3             4-methylheptane        CCCC(C)CCC  117.7            -50.69   
4          2,2-dimethylhexane      CC(C)(C)CCCC  106.8            -53.71   
5          2,3-dimethylhexane      CC(C)C(C)CCC  115.6            -51.13   
6          2,4-dimethylhexane      CC(C)CC(C)CC  109.4            -52.44   
7          2,5-dimethylhexane      CC(C)CCC(C)C  109.1            -53.21   
8          3,3-dimethylhexane      CCCC(C)(C)CC  111.9            -52.61   
9          3,4-dimethylhexane      CCC(C)C(C)CC  117.7            -50.91   
10    3-ethyl-2-methylpentane      CCC(CC)C(C)C  115.6            -50.48   
11    3-ethyl-3-methylpentane      CCC(C)(CC)CC  118.3            -51.38   
12          

True

### Nonane isomers (Section 6.3) and measured boiling points (Section 6.4)

In [5]:
non = pd.read_csv("DATA/nonane_properties.csv")
bp = pd.read_csv("DATA/boiling_points_C6_C10.csv")
print(non.head()); print(bp.head())
check("nonane isomers", 35, len(non))
check("nonane dHvap values", 34, int(non.dHvap_kcal.notna().sum()))
check("pooled C6-C10 boiling points", 100, len(bp))
check("decanes in the boiling-point cohort", 34, int((bp.n_C == 10).sum()))

             name         cas       smiles  T_B_C  dHvap_kcal
0          nonane    111-84-2    CCCCCCCCC  150.7       11.11
1  2-methyloctane   3221-61-2  CC(C)CCCCCC  143.0       10.73
2  3-methyloctane   2216-33-3  CCC(C)CCCCC  143.9       10.73
3  4-methyloctane   2216-34-4  CCCC(C)CCCC  141.9       10.64
4  3-ethylheptane  15869-80-4  CCC(CC)CCCC  143.6       10.64
   n_C                   name  nist_id  T_B_C         degree_pairs
0    6  Butane, 2,3-dimethyl-   C79298   58.1  1-3 1-3 1-3 1-3 3-3
1    6     Pentane, 2-methyl-  C107835   60.9  1-2 1-3 1-3 2-2 2-3
2    6     Pentane, 3-methyl-   C96140   63.3  1-2 1-2 1-3 2-3 2-3
3    6  Butane, 2,2-dimethyl-   C75832   49.8  1-2 1-4 1-4 1-4 2-4
4    6               n-Hexane  C110543   68.8  1-2 1-2 2-2 2-2 2-2
OK       nonane isomers: paper 35, recomputed 35
OK       nonane dHvap values: paper 34, recomputed 34
OK       pooled C6-C10 boiling points: paper 100, recomputed 100
OK       decanes in the boiling-point cohort: paper 34, re

True

### Provenance

Every value has a source record. The tetramethylbutane audit is in `TMB_DHVAP_AUDIT.md`.

In [6]:
prov = pd.read_csv("DATA/octane_provenance.csv")
print(prov.classification.value_counts().to_string())
print(open("TMB_DHVAP_AUDIT.md").read()[:1500])

classification
AGREEMENT WITHIN TOLERANCE        40
VERIFIED AFTER UNIT CONVERSION    29
SOURCE VARIATION / EXPLAINED      10
CANNOT VERIFY                      7
VERIFIED EXACT                     3
CONFLICT                           1
# Source audit: the vaporisation enthalpy of 2,2,3,3-tetramethylbutane

The octane benchmark used in Table 4 of the paper gives
ΔH_vap = 8.410 kcal/mol for 2,2,3,3-tetramethylbutane (TMB, CAS 594-82-1).
NIST lists ΔvapH° = 42.94 and 42.91 kJ/mol (10.26 kcal/mol). This file traces
both values and records what each one measures. The benchmark value is kept
unchanged in the paper; the alternatives are sensitivity analyses only.

## 1. Phase behaviour

* NIST WebBook (archived copy: `external/bp/nist_raw/C594821.html`): triple point
  373.97 K (Scott, Douslin et al. 1952), T_boil 379.6 K, T_fus 374 K.
  TMB is therefore crystalline at 298 K. A liquid at 298 K exists only as a
  hypothetical supercooled state.

## 2. Values on record

| Quantity | Value | Te

## Summary

In [7]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                                check   paper recomputed status
0           script: python DATA/build_data.py --check  exit 0     exit 0     OK
1                            number of octane isomers      18         18     OK
2  dHvap of 2,2,3,3-tetramethylbutane used (kcal/mol)    8.41       8.41     OK
3                                      nonane isomers      35         35     OK
4                                 nonane dHvap values      34         34     OK
5                        pooled C6-C10 boiling points     100        100     OK
6                 decanes in the boiling-point cohort      34         34     OK
7 checks, 0 mismatches
